# 02 — Bronze avec Auto Loader

Lit les pages JSON déposées dans le Volume `landing/erp` et les ajoute dans trois tables Delta :
`bronze.erp_suppliers`, `bronze.erp_parts`, `bronze.erp_purchase_orders`.

- **une ligne par enregistrement**, JSON brut conservé dans `payload` ;
- colonnes de traçabilité : `_run_id`, `_fetched_at`, `_file`, `_ingest_ts` ;
- **doublons conservés** : le dédoublonnage est la première règle du Silver ;
- **incrémental** : le checkpoint mémorise les fichiers déjà lus, une nouvelle exécution ne lit que les nouveaux fichiers ;
- `trigger(availableNow=True)` : traite tout ce qui est nouveau puis s'arrête (aucun calcul qui tourne à vide).

In [ ]:
# Cellule 1 - paramètres
dbutils.widgets.text("catalog", "supplychain")
dbutils.widgets.dropdown("reset", "non", ["non", "oui"])   # "oui" = reconstruire le Bronze de zéro

CATALOG = dbutils.widgets.get("catalog")
RESET = dbutils.widgets.get("reset") == "oui"

LANDING = f"/Volumes/{CATALOG}/landing/erp"
CHECKPOINTS = f"/Volumes/{CATALOG}/landing/checkpoints"

# entité de l'API -> table Bronze
TABLES = {"suppliers": "erp_suppliers",
          "parts": "erp_parts",
          "purchase-orders": "erp_purchase_orders"}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"USE CATALOG {CATALOG}")                      # les cellules SQL écrivent bronze.xxx
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.checkpoints")

## Étape 0 — Inventaire du Volume

Avant de charger : quels runs sont présents, et combien de pages chacun.
Un run inutile (relancé depuis la page 1 par erreur, test…) se supprime **maintenant** :
une fois lu par Auto Loader, il reste dans le Bronze.

In [ ]:
# Cellule 2 - inventaire : entité / run / nombre de pages
inventory = []
for entity in TABLES:
    for day in dbutils.fs.ls(f"{LANDING}/{entity}"):            # ingest_date=AAAAMMJJ/
        for run in dbutils.fs.ls(day.path):                     # run_id=…/
            pages = [f for f in dbutils.fs.ls(run.path) if f.name.startswith("page_")]
            inventory.append((entity, run.name.strip("/").removeprefix("run_id="), len(pages)))

display(spark.createDataFrame(inventory, "entite STRING, run_id STRING, pages INT")
        .orderBy("entite", "run_id"))

In [ ]:
# Cellule 3 - remise à zéro (seulement si le widget reset = "oui")
# Table ET checkpoint ensemble : supprimer l'un sans l'autre donne une table vide ou des doublons.
if RESET:
    for name in TABLES.values():
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.bronze.{name}")
        dbutils.fs.rm(f"{CHECKPOINTS}/{name}", recurse=True)
        print(f"{name} : table et checkpoint supprimés")
else:
    print("reset = non : chargement incrémental")

## Étape 1 — Chargement incrémental

In [ ]:
# Cellule 4 - la fonction de chargement d'une entité
import time
from pyspark.sql import functions as F

# Schéma volontairement minimal et déclaré : pas d'inférence (aucune lecture préalable des fichiers),
# chaque élément de data reste un JSON brut (string), les dates mal formées ne sont pas perdues.
PAGE_SCHEMA = "data ARRAY<STRING>, _run_id STRING, _fetched_at STRING"


def row_count(table: str) -> int:
    return spark.table(table).count() if spark.catalog.tableExists(table) else 0


def autoload_entity(entity: str, table_name: str) -> None:
    table = f"{CATALOG}.bronze.{table_name}"
    before = row_count(table)
    start = time.time()

    pages = (spark.readStream
             .format("cloudFiles")                       # Auto Loader
             .option("cloudFiles.format", "text")
             .option("wholetext", True)                  # un fichier = une page = une ligne
             .option("pathGlobFilter", "page_*.json")    # seulement les pages
             .option("recursiveFileLookup", True)        # parcourt ingest_date=…/run_id=…
             .load(f"{LANDING}/{entity}"))

    records = (pages
               .select(F.from_json("value", PAGE_SCHEMA).alias("p"),
                       F.col("_metadata.file_path").alias("_file"))
               .select("p._run_id", "p._fetched_at", "_file",
                       F.explode("p.data").alias("payload"))       # une ligne par enregistrement
               .select(F.get_json_object("payload", "$.id").alias("source_id"),
                       F.get_json_object("payload", "$.audit.updated_at").alias("source_updated_at"),
                       "payload", "_run_id", "_fetched_at", "_file",
                       F.current_timestamp().alias("_ingest_ts")))

    query = (records.writeStream
             .queryName(f"bronze_{table_name}")
             .option("checkpointLocation", f"{CHECKPOINTS}/{table_name}")
             .trigger(availableNow=True)                 # tout ce qui est nouveau, puis arrêt
             .toTable(table))
    query.awaitTermination()

    n_pages = sum(p["numInputRows"] for p in query.recentProgress)   # entrées = fichiers (wholetext)
    added = row_count(table) - before
    print(f"{table_name} : {n_pages} pages lues, {added} lignes ajoutées, "
          f"{row_count(table)} au total, {time.time() - start:.0f} s")

In [ ]:
# Cellule 5 - chargement des trois tables
for entity, name in TABLES.items():
    autoload_entity(entity, name)

## Étape 2 — Contrôles

À comparer avec la zone de dépôt (exploration DuckDB et inventaire de l'étape 0) :

| Run | Lignes attendues |
| --- | --- |
| un run fournisseurs | 21 600 |
| run articles | 399 958 |
| commandes `20261101T120141Z-39b653` | 872 000 |
| commandes `20261102T105626Z-30c1cc` (reprise) | 418 380 |

In [ ]:
%sql
-- Cellule 6 - volumes par table
SELECT 'erp_suppliers' AS table_bronze, count(*) AS lignes, count(DISTINCT source_id) AS ids_distincts,
       count(DISTINCT _run_id) AS runs, count(DISTINCT _file) AS pages
FROM bronze.erp_suppliers
UNION ALL
SELECT 'erp_parts', count(*), count(DISTINCT source_id), count(DISTINCT _run_id), count(DISTINCT _file)
FROM bronze.erp_parts
UNION ALL
SELECT 'erp_purchase_orders', count(*), count(DISTINCT source_id), count(DISTINCT _run_id), count(DISTINCT _file)
FROM bronze.erp_purchase_orders

In [ ]:
%sql
-- Cellule 7 - lignes et pages par run (doit correspondre à l'inventaire de l'étape 0)
SELECT 'suppliers' AS entite, _run_id, count(*) AS lignes, count(DISTINCT _file) AS pages
FROM bronze.erp_suppliers GROUP BY _run_id
UNION ALL
SELECT 'parts', _run_id, count(*), count(DISTINCT _file) FROM bronze.erp_parts GROUP BY _run_id
UNION ALL
SELECT 'purchase-orders', _run_id, count(*), count(DISTINCT _file) FROM bronze.erp_purchase_orders GROUP BY _run_id
ORDER BY entite, _run_id

In [ ]:
%sql
-- Cellule 8 - qualité technique du Bronze : tout doit valoir 0
SELECT 'erp_suppliers' AS table_bronze,
       count_if(payload IS NULL)           AS payload_vide,
       count_if(source_id IS NULL)         AS sans_id,
       count_if(source_updated_at IS NULL) AS sans_updated_at,
       count_if(_run_id IS NULL)           AS sans_run
FROM bronze.erp_suppliers
UNION ALL
SELECT 'erp_parts', count_if(payload IS NULL), count_if(source_id IS NULL),
       count_if(source_updated_at IS NULL), count_if(_run_id IS NULL) FROM bronze.erp_parts
UNION ALL
SELECT 'erp_purchase_orders', count_if(payload IS NULL), count_if(source_id IS NULL),
       count_if(source_updated_at IS NULL), count_if(_run_id IS NULL) FROM bronze.erp_purchase_orders

## Étape 3 — Preuve d'idempotence

Relancer la **cellule 5** : chaque table doit afficher `0 pages lues, 0 lignes ajoutées`.
Le checkpoint a mémorisé tous les fichiers ; relancer ne coûte presque rien et ne crée aucun doublon.

L'historique Delta montre une écriture (`STREAMING UPDATE`) par micro-lot :

In [ ]:
%sql
-- Cellule 9 - historique des écritures
DESCRIBE HISTORY bronze.erp_purchase_orders

## FinOps

Le notebook se termine de lui-même (`availableNow`) : aucun flux ne reste ouvert.
Une fois les contrôles faits, **détacher le notebook** ou arrêter le calcul, puis vérifier la consommation
dans `system.billing.usage` le lendemain.